In [2]:
import pandas as pd
import numpy as np
import warnings
import copy
import pickle
from sklearn.preprocessing import MinMaxScaler,LabelEncoder
warnings.filterwarnings('ignore')

In [18]:
class MGFS:
    
    def __init__(self,data):
        self.data = data
        self.attr_names = list(self.data)[:-1]
        self.attr_num = len(self.attr_names)
        self.attr_list = [x for x in range(self.attr_num)]
        self.sample_num = len(self.data)
        self.class_name = list(self.data)[-1]
        self.class_num = len(list(self.data[self.class_name].value_counts().index))
        self.D_mat = None
        self.fuzzy_list = np.zeros((self.attr_num,self.sample_num,self.sample_num))
        self.judge_attr = [[] for x in range(self.attr_num)]
        self.Imp_list = [[] for x in range(self.attr_num)]
        self.Dep_list = [[] for x in range(self.attr_num)]
        self.feature_sort = None
        self.res = [[] for x in range(11)]
    
    """macro-granular interactive mutual information"""
    def IMI(self,f_mat):
        U_mat = abs(f_mat-self.D_mat)
        f_IMI = -np.mean(np.log(np.mean(U_mat,axis=1)))
        return f_IMI
    
    """dependency"""
    def Dep(self,B):
        if(len(B)==1):
            B_mat = self.fuzzy_list[B[0]]
        else:
            B_mat = np.minimum.reduce(self.fuzzy_list[B])
        lower_appro = np.zeros((self.sample_num,self.class_num))
        for i,d_name in enumerate(self.class_value):
            mask_array = self.class_mask_dict[d_name]
            crisp_D = np.broadcast_to(mask_array,(self.sample_num,self.sample_num))
            fuzzy_D = np.broadcast_to(np.sum(np.minimum(B_mat,crisp_D),axis=1)/np.sum(B_mat,axis=1),(self.sample_num,self.sample_num))
            lower_approximations = np.min(np.maximum(1-B_mat,fuzzy_D),axis=1)
            lower_appro[:,i] = lower_approximations
        POS_B_D = np.max(lower_appro,axis=1)
        Dep_B_D = np.mean(POS_B_D)
        return Dep_B_D
    
    """macro-granular conditional entropy"""
    def CMGE(self,B):
        if(len(B)==1):
            B_mat = self.fuzzy_list[B[0]]
        else:
            B_mat = np.minimum.reduce(self.fuzzy_list[B])
        f_CMGE = -np.mean(np.log(np.sum(self.D_mat,axis=1)/np.sum(np.maximum(self.D_mat,B_mat))))
        return f_CMGE
    
    def data_deal(self):
        self.data[self.class_name] = LabelEncoder().fit_transform(self.data[self.class_name])
        vector_d = self.data[[self.class_name]].values
        self.D_mat = (vector_d==vector_d.T).astype(int)
        self.class_value = np.array(self.data[self.class_name].value_counts().index)
        self.class_v_num = np.array(self.data[self.class_name].value_counts())
        self.class_mask_dict = np.zeros((self.class_num,self.sample_num))
        for d in self.class_value:
            self.class_mask_dict[d] = (self.data[self.class_name]==d).values
        #feature dimensions greater than 100 default to numeric data
        if(self.attr_num>100):
            self.data[self.attr_names] = MinMaxScaler().fit_transform(self.data[self.attr_names])
            self.judge_attr = [1 for x in range(self.attr_num)]
        else:
            for i,name in enumerate(self.attr_names):
                if (np.issubdtype(self.data[name],np.number)==True):
                    self.data[[name]] = MinMaxScaler().fit_transform(self.data[[name]])
                    self.judge_attr[i] = 1
                else:
                    vector = self.data.values[:,i][:,np.newaxis]
                    self.fuzzy_list[i] = (vector==vector.T).astype(int)
                    f_IMI = self.IMI(f_mat=self.fuzzy_list[i])
                    f_Dep = self.Dep(B=[i])
                    self.Dep_list[i] = f_Dep
                    f_Imp = f_IMI*f_Dep
                    self.Imp_list[i] = f_Imp
                    self.judge_attr[i] = 0
    
    #Construct multi-granularity fuzzy granules and calculate the feature evaluation metrics
    def multi_granularity(self,lanbda):
        for i,judge in enumerate(self.judge_attr):
            if(judge==1):
                radius = np.std(self.data.values[:,i])/lanbda
                vector = self.data.values[:,i][:,np.newaxis]
                fuzzy_mat = 1-abs(vector-vector.T)
                fuzzy_mat[fuzzy_mat<np.array([1-radius])] = 0
                self.fuzzy_list[i] = np.float32(fuzzy_mat)
                f_IMI = self.IMI(f_mat=self.fuzzy_list[i])
                f_Dep = self.Dep(B=[i])
                self.Dep_list[i] = f_Dep
                f_Imp = f_IMI*f_Dep
                self.Imp_list[i] = f_Imp
            
    def feature_selection(self,mu):
        """feature ranking sequence"""
        self.feature_sort = list(np.argsort(self.Imp_list)[::-1])[:mu]
        Dep_B = self.Dep_list[np.argmax(self.Imp_list)]
        CMGE_B = self.CMGE(B=[self.feature_sort[0]])
        index = 1
        for i in range(mu-2):
            B_1 = copy.deepcopy(self.feature_sort[:index])
            B_2 = copy.deepcopy(self.feature_sort[:index])
            B_1.append(self.feature_sort[index])
            B_2.append(self.feature_sort[index+1])
            Sig_1 = self.Dep(B=B_1)-Dep_B
            Urd_1 = CMGE_B-self.CMGE(B=B_1)
            Sig_2 = self.Dep(B=B_2)-Dep_B
            Urd_2 = CMGE_B-self.CMGE(B=B_2)
            """local position fine-tuning"""
            ind_1 = int(Sig_2>Sig_1)
            ind_2 = int(Urd_2>Urd_1)
            ind = ind_1+ind_2
            if(ind==2):
                self.feature_sort[index],self.feature_sort[index+1] = self.feature_sort[index+1],self.feature_sort[index]
                Dep_B = copy.deepcopy(Sig_2+Dep_B)
                CMGE_B = copy.deepcopy(CMGE_B-Urd_2)
            else:
                Dep_B = copy.deepcopy(Sig_1+Dep_B)
                CMGE_B = copy.deepcopy(CMGE_B-Urd_1)
            index += 1
            
    def run(self):
        self.data_deal()
        mu = None
        if self.attr_num>1000:
            mu = 50
        else:
            mu = self.attr_num
        param_list = [0.5,0.6,0.7,0.8,0.9,1,1.1,1.2,1.3,1.4,1.5]
        for count,param in enumerate(param_list):
            print('lambda: {}'.format(param))
            self.multi_granularity(lanbda=param)
            self.feature_selection(mu=mu)
            self.res[count] = copy.deepcopy(self.feature_sort)
            print('S: {}'.format(self.feature_sort))
            self.feature_sort = None

In [ ]:
df = pd.read_csv('')
# If nominal features are present in the data and are numerically encoded, be sure to use object conversion.
# For example:
df[['nominal_1']] = df[['nominal_1']].astype('object')

In [ ]:
model = MGFS(data=df)
model.run()

In [ ]:
print(self.res)

In [17]:
del model